<a href="https://colab.research.google.com/github/MarceCorreal2/Robots-NT/blob/main/Procesamiento_Indicadores_Backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Este cuaderno limpia y organiza los datos del resumen del strategy analyzer de los robots y debe incluir los indicadores en la
# tabla robots y procesar la calificación

## Procesamiento de Indicadores de Backtest

El objetivo de este cuaderno es tomar los datos crudos de los resúmenes del *strategy analyzer* de tus robots, limpiarlos y extraer los indicadores clave mencionados. Una vez procesados, los datos se guardarán en un formato limpio para futuros análisis.

In [14]:
# Celda 1 — Importes y configuración

import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [15]:
# Celda 2 — Conectar Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Celda 3 - Establecer Variables

bot_name = 'RB01_MNQ_A'
test_number = 'T001' # <--- Modifica este valor para diferentes pruebas (ej. 'T002')
instrument = 'MNQ'

print(f"Bot Name: {bot_name}")
print(f"Test Number: {test_number}")
print(f"Instrument: {instrument}")

Bot Name: RB01_MNQ_A
Test Number: T001
Instrument: MNQ


In [17]:
# Celda 4 - Definir Rutas de Datos

RAW_DATA_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/'
CLEAN_DATA_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/'

print(f"Ruta de Datos Crudos: {RAW_DATA_PATH}")
print(f"Ruta de Datos Limpios: {CLEAN_DATA_PATH}")

Ruta de Datos Crudos: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/
Ruta de Datos Limpios: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/


In [19]:
# Celda 5 - Cargar y mostrar un archivo CSV de ejemplo con parsing robusto

import os
import pandas as pd

sample_file_name = 'SA-RB01-MNQ-A-T001.csv'
sample_file_path = os.path.join(RAW_DATA_PATH, sample_file_name)

print(f"Cargando archivo de ejemplo: {sample_file_path}")

try:
    # Read the first few lines to understand the structure and find the actual data start
    raw_lines = []
    with open(sample_file_path, 'r', encoding='latin1') as f:
        for _ in range(30): # Read up to 30 lines to cover potential header length
            line = f.readline()
            if not line: # EOF
                break
            raw_lines.append(line.strip())

    print("\nPrimeras 20 líneas del archivo crudo para inspección:")
    for i, line in enumerate(raw_lines[:20]):
        print(f"Línea {i+1}: {line}")

    # Try to find the line that indicates the start of the actual performance metrics
    # Common indicators like 'Total net profit' usually appear at the start of data section.
    data_start_row = -1
    for i, line in enumerate(raw_lines):
        if 'Total net profit' in line:
            data_start_row = i
            break

    if data_start_row != -1:
        print(f"\nIdentificado el inicio de los datos de indicadores en la línea (0-index): {data_start_row}")
        # Read the CSV again, skipping lines up to the identified data start
        # We set header=None because the first column will contain the indicator names,
        # and the subsequent columns are values (e.g., All trades, Long trades, Short trades).
        sample_df = pd.read_csv(sample_file_path, encoding='latin1', sep=';', skiprows=data_start_row, header=None)

        # Assuming the first column is the indicator name and the next are its values
        # We need to clean up the column names based on the context.
        # Let's just display the raw parsed DataFrame for now.
        print("\nDataFrame de indicadores procesado (primeras 5 filas):")
        print(sample_df.head().to_markdown(index=False, numalign="left", stralign="left"))
        print("\nColumnas del DataFrame procesado:")
        print(sample_df.columns.tolist())
    else:
        print("\nNo se pudo identificar el inicio de los datos de indicadores ('Total net profit' no encontrado). Se muestra la lectura inicial sin procesar.")
        # Fallback if specific data start not found, try reading with just separator
        sample_df = pd.read_csv(sample_file_path, encoding='latin1', sep=';')
        print("\nPrimeras 5 filas del archivo de ejemplo (lectura básica):")
        print(sample_df.head().to_markdown(index=False, numalign="left", stralign="left"))
        print("\nColumnas del archivo de ejemplo (lectura básica):")
        print(sample_df.columns.tolist())

except FileNotFoundError:
    print(f"Error: El archivo '{sample_file_name}' no se encontró en la ruta '{RAW_DATA_PATH}'. Por favor, verifica la ruta y el nombre del archivo.")
except Exception as e:
    print(f"Error al leer el archivo CSV: {e}")

Cargando archivo de ejemplo: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/SA-RB01-MNQ-A-T001.csv

Primeras 20 líneas del archivo crudo para inspección:
Línea 1: Performance;All trades;Long trades;Short trades;
Línea 2: Total net profit;$ 213,00;$ 213,00;$ 0,00;
Línea 3: Gross profit;$ 2243,00;$ 2243,00;$ 0,00;
Línea 4: Gross loss;-$ 2030,00;-$ 2030,00;$ 0,00;
Línea 5: Commission;$ 95,00;$ 95,00;$ 0,00;
Línea 6: Profit factor;1,10;1,10;1,00;
Línea 7: Max drawdown;-$ 995,60;-$ 995,60;$ 0,00;
Línea 8: Sharpe ratio;2,29;2,29;1,00;
Línea 9: Sortino ratio;1,00;1,00;1,00;
Línea 10: Ulcer index;0,01;0,01;0,00;
Línea 11: R squared;0,18;0,18;0,00;
Línea 12: Total Fees;$ 0,00;$ 0,00;$ 0,00;
Línea 13: Probability;42,14 %;42,14 %;0,00 %;
Línea 14: ;;;;
Línea 15: Start date;25/05/2026;;;
Línea 16: Start time;12:00 AM;;;
Línea 17: End date;3/06/2026;;;
Línea 18: End time;12:00 AM;;;
Línea 19: ;;;;
Línea 20: Total # of trades;50;50;0;

Identificado el inicio de los datos 

In [20]:
# Celda 6 - Extracción indicadores

import os
import pandas as pd
import re
from datetime import datetime

# Lista para almacenar los DataFrames de indicadores de cada archivo
all_indicators_list = []

# Función para limpiar y convertir valores numéricos
def clean_numeric_value(value):
    if isinstance(value, str):
        value = value.replace('$', '').replace(' ', '').replace('%', '').replace(',', '.')
        if value == '' or value == '-':
            return None
        try:
            return float(value)
        except ValueError:
            return value
    return value

# Iterar sobre todos los archivos en la carpeta de datos crudos
print(f"Procesando archivos en: {RAW_DATA_PATH}")
for file_name in os.listdir(RAW_DATA_PATH):
    if file_name.endswith('.csv'):
        full_file_path = os.path.join(RAW_DATA_PATH, file_name)
        print(f"\n--- Procesando archivo: {file_name} ---")

        try:
            # --- Parsing robusto (similar a Celda 5) ---
            raw_lines = []
            with open(full_file_path, 'r', encoding='latin1') as f:
                for _ in range(30): # Read up to 30 lines to find data start
                    line = f.readline()
                    if not line: break
                    raw_lines.append(line.strip())

            data_start_row = -1
            for i, line in enumerate(raw_lines):
                if 'Total net profit' in line:
                    data_start_row = i
                    break

            if data_start_row == -1:
                print(f"Advertencia: No se encontró el inicio de datos para {file_name}. Saltando este archivo.")
                continue

            temp_df = pd.read_csv(full_file_path, encoding='latin1', sep=';', skiprows=data_start_row, header=None)

            # --- Limpieza y extracción (similar a Celda 6) ---
            # Aplicar nombres de columna iniciales y establecer índice
            temp_df.columns = ['Performance', 'All trades', 'Long trades', 'Short trades', 'Extra_Column']
            temp_df = temp_df.drop(columns=['Extra_Column'])
            temp_df['Performance'] = temp_df['Performance'].str.strip()
            temp_df = temp_df.set_index('Performance')

            # Aplicar limpieza a valores numéricos
            for col in ['All trades', 'Long trades', 'Short trades']:
                temp_df[col] = temp_df[col].apply(clean_numeric_value)

            def get_indicator_value(df, indicator_name):
                try:
                    return df.loc[indicator_name.strip(), 'All trades']
                except KeyError:
                    return None

            # Extraer los indicadores solicitados
            NetProfit = get_indicator_value(temp_df, 'Total net profit')
            PF = get_indicator_value(temp_df, 'Profit factor')
            WR_probability = get_indicator_value(temp_df, 'Probability')
            DD_max = get_indicator_value(temp_df, 'Max drawdown')
            RecoveryFactor = get_indicator_value(temp_df, 'Recovery factor')
            TotalTrades = get_indicator_value(temp_df, 'Total # of trades')
            Winners = get_indicator_value(temp_df, 'Number of winning trades')
            GrossProfit = get_indicator_value(temp_df, 'Gross profit')
            GrossLoss = get_indicator_value(temp_df, 'Gross loss')
            AvgWinTrade = get_indicator_value(temp_df, 'Avg winning trade') # New indicator
            AvgLossTrade = get_indicator_value(temp_df, 'Avg losing trade') # New indicator

            WR = WR_probability if WR_probability is not None else \
                 (Winners / TotalTrades) * 100 if TotalTrades and Winners is not None and TotalTrades != 0 else None

            PayoffRatio = abs(GrossProfit / GrossLoss) if GrossProfit and GrossLoss and GrossLoss != 0 else None

            # Extracción de fechas y cálculo de Net Profit/Mes
            FechaInicio = None
            FechaFin = None
            for line in raw_lines:
                if 'Start date' in line:
                    match = re.search(r'Start date;(\d{1,2}/\d{1,2}/\d{4});', line)
                    if match:
                        FechaInicio = datetime.strptime(match.group(1), '%d/%m/%Y')
                elif 'End date' in line:
                    match = re.search(r'End date;(\d{1,2}/\d{1,2}/\d{4});', line)
                    if match:
                        FechaFin = datetime.strptime(match.group(1), '%d/%m/%Y')

            NumMonths = None
            NetProfitPerMonth = None
            if NetProfit is not None and FechaInicio is not None and FechaFin is not None:
                delta = FechaFin - FechaInicio
                if delta.days > 0:
                    NumMonths = delta.days / 30.44
                    if NumMonths > 0:
                        NetProfitPerMonth = NetProfit / NumMonths

            # Extraer Robot Base, Test ID e Instrumento del nombre del archivo
            # Ejemplo: SA-RB01-MNQ-A-T001.csv
            file_parts = file_name.replace('.csv', '').split('-')
            robot_base = file_parts[1] if len(file_parts) > 1 else None
            test_id = file_parts[-1] if len(file_parts) > 0 else None # Assuming last part is Test ID
            instrument_from_file = file_parts[2] if len(file_parts) > 2 else None

            # Crear un diccionario con los indicadores para este archivo
            file_indicators = {
                'Archivo': file_name,
                'Robot Base': robot_base,
                'Test ID': test_id,
                'Instrumento': instrument_from_file,
                'NetProfit': NetProfit,
                'PF': PF,
                'WR': WR,
                'DD max': DD_max,
                'Recovery Factor': RecoveryFactor,
                'PayoffRatio': PayoffRatio,
                '# Trades': TotalTrades,
                '# Meses': NumMonths,
                'Net Profit/Mes': NetProfitPerMonth,
                'Fecha-Inicio': FechaInicio.strftime('%Y-%m-%d') if FechaInicio else None,
                'Fecha-Fin': FechaFin.strftime('%Y-%m-%d') if FechaFin else None,
                'Avg Win': AvgWinTrade, # New indicator added
                'Avg Loss': AvgLossTrade # New indicator added
            }
            all_indicators_list.append(file_indicators)

        except Exception as e:
            print(f"Error al procesar el archivo {file_name}: {e}")

# Convertir la lista de diccionarios a un DataFrame consolidado
if all_indicators_list:
    consolidated_df = pd.DataFrame(all_indicators_list)
    print("\n--- DataFrame Consolidado de Indicadores (primeras 5 filas): ---")
    display(consolidated_df.head())

    # Guardar el DataFrame consolidado en un nuevo archivo CSV
    output_file_path = os.path.join(CLEAN_DATA_PATH, 'backtest_indicators_summary.csv')
    consolidated_df.to_csv(output_file_path, index=False)
    print(f"\nDataFrame consolidado guardado en: {output_file_path}")
else:
    print("No se pudieron procesar indicadores de ningún archivo.")

Procesando archivos en: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/

--- Procesando archivo: SA-RB01-MNQ-A-T001.csv ---

--- DataFrame Consolidado de Indicadores (primeras 5 filas): ---


,Archivo,Robot Base,Test ID,Instrumento,NetProfit,PF,WR,DD max,Recovery Factor,PayoffRatio,# Trades,# Meses,Net Profit/Mes,Fecha-Inicio,Fecha-Fin,Avg Win,Avg Loss
0,SA-RB01-MNQ-A-T001.csv,RB01,T001,MNQ,213.0,1.1,42.14,-995.6,None,1.104926,50.0,0.295664,720.413333,2026-05-25,2026-06-03,224.3,-50.75



DataFrame consolidado guardado en: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/backtest_indicators_summary.csv


In [22]:
# Celda 7 - Limpieza y reordenamiento de datos finales

# Renombrar columnas para que coincidan exactamente con la solicitud del usuario
consolidated_df = consolidated_df.rename(columns={
    'Fecha-Inicio': 'Fecha Inicio',
    'Fecha-Fin': 'Fecha Fin',
    'DD max': 'DD Max',
    'Net Profit/Mes': 'Net Profit/Mes'
})

# Definir el orden de las columnas solicitado por el usuario
column_order = [
    'Fecha Inicio',
    'Fecha Fin',
    'Instrumento',
    '# Meses',
    '# Trades',
    'NetProfit',
    'Net Profit/Mes',
    'PF',
    'WR',
    'DD Max',
    'Recovery Factor',
    'PayoffRatio',
    'Avg Win',
    'Avg Loss'
]

# Seleccionar y reordenar las columnas del DataFrame
# Se maneja si alguna columna esperada no está presente, aunque con las modificaciones anteriores deberían estar todas.
final_df = consolidated_df[column_order]

print("\n--- DataFrame Final con Columnas Limpias y Reordenadas (primeras 5 filas): ---")
display(final_df.head())


--- DataFrame Final con Columnas Limpias y Reordenadas (primeras 5 filas): ---


,Fecha Inicio,Fecha Fin,Instrumento,# Meses,# Trades,NetProfit,Net Profit/Mes,PF,WR,DD Max,Recovery Factor,PayoffRatio,Avg Win,Avg Loss
0,2026-05-25,2026-06-03,MNQ,0.295664,50.0,213.0,720.413333,1.1,42.14,-995.6,None,1.104926,224.3,-50.75


In [25]:
# Celda 8 - Verificar tipos de datos del DataFrame final
print("\n--- Tipos de datos del DataFrame final: ---")
display(final_df.dtypes)


--- Tipos de datos del DataFrame final: ---


,0
Fecha Inicio,object
Fecha Fin,object
Instrumento,object
# Meses,float64
# Trades,float64
NetProfit,float64
Net Profit/Mes,float64
PF,float64
WR,float64
DD Max,float64
